In [18]:
import pandas as pd

routes = pd.read_csv("gtfs/routes.txt", dtype=str)
trips = pd.read_csv("gtfs/trips.txt", dtype=str)
stops = pd.read_csv("gtfs/stops.txt", dtype=str)

routes.head()


,route_id,agency_id,route_short_name,route_long_name,route_type,route_url,route_color,route_text_color,route_desc,route_desc_detail
0,1,STM,1,Ligne 1 - Verte,1,https://www.stm.info/fr/infos/reseaux/metro/li...,00B300,FFFFFF,NaN,NaN
1,2,STM,2,Ligne 2 - Orange,1,https://www.stm.info/fr/infos/reseaux/metro/li...,D95700,FFFFFF,NaN,NaN
2,4,STM,4,Ligne 4 - Jaune,1,https://www.stm.info/fr/infos/reseaux/metro/li...,FFD900,000000,NaN,NaN
3,5,STM,5,Ligne 5 - Bleue,1,https://www.stm.info/fr/infos/reseaux/metro/li...,0095E6,FFFFFF,NaN,NaN
4,10,STM,10,De Lorimier,3,https://www.stm.info/fr/infos/reseaux/bus,009EE0,FFFFFF,Jour,Lignes de jour seulement


In [19]:
metro_routes = routes[routes["route_type"] == "1"]
metro_routes

,route_id,agency_id,route_short_name,route_long_name,route_type,route_url,route_color,route_text_color,route_desc,route_desc_detail
0,1,STM,1,Ligne 1 - Verte,1,https://www.stm.info/fr/infos/reseaux/metro/li...,00B300,FFFFFF,NaN,NaN
1,2,STM,2,Ligne 2 - Orange,1,https://www.stm.info/fr/infos/reseaux/metro/li...,D95700,FFFFFF,NaN,NaN
2,4,STM,4,Ligne 4 - Jaune,1,https://www.stm.info/fr/infos/reseaux/metro/li...,FFD900,000000,NaN,NaN
3,5,STM,5,Ligne 5 - Bleue,1,https://www.stm.info/fr/infos/reseaux/metro/li...,0095E6,FFFFFF,NaN,NaN


In [20]:
metro_trips = trips[trips["route_id"].isin(metro_routes["route_id"])]
len(metro_trips)

12668

In [21]:
metro_trips.head()

,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id,wheelchair_accessible,route_pattern_id
0,1,26S-GLOBAUX-01-S,299969299,Station Honoré-Beaugrand,0,1_1071,1,1_1071
1,1,26S-GLOBAUX-01-S,299969302,Station Angrignon,1,1_1072,1,1_1072
2,1,26S-GLOBAUX-01-S,299969305,Station Honoré-Beaugrand,0,1_1071,1,1_1071
3,1,26S-GLOBAUX-01-S,299969308,Station Angrignon,1,1_1072,1,1_1072
4,1,26S-GLOBAUX-01-S,299969311,Station Honoré-Beaugrand,0,1_1071,1,1_1071


In [22]:
metro_trip_ids = set(metro_trips["trip_id"])

chunks = pd.read_csv("gtfs/stop_times.txt", dtype=str, chunksize=500_000)
metro_stop_times = pd.concat(
    chunk[chunk["trip_id"].isin(metro_trip_ids)] for chunk in chunks
)

len(metro_stop_times)

237804

In [23]:
# Pick the first Orange line trip (route_id "2")
orange_trip = metro_trips[metro_trips["route_id"] == "2"].iloc[0]
print(orange_trip["trip_id"], "→", orange_trip["trip_headsign"])

# Get its stops, in order
one_trip = metro_stop_times[metro_stop_times["trip_id"] == orange_trip["trip_id"]].copy()
one_trip["stop_sequence"] = one_trip["stop_sequence"].astype(int)
one_trip = one_trip.sort_values("stop_sequence")

# Attach station names
one_trip = one_trip.merge(stops[["stop_id", "stop_name"]], on="stop_id")

one_trip[["stop_sequence", "stop_name", "arrival_time", "departure_time"]]

299970640 → Station Côte-Vertu


,stop_sequence,stop_name,arrival_time,departure_time
0,1,Station Montmorency -Zone B,05:54:00,05:54:00
1,2,Station De la Concorde -Zone B,05:55:00,05:55:00
2,3,Station Cartier -Zone B,05:58:00,05:58:00
3,4,Station Henri-Bourassa,05:59:00,05:59:00
4,5,Station Sauvé,06:01:00,06:01:00
5,6,Station Crémazie,06:03:00,06:03:00
6,7,Station Jarry,06:05:00,06:05:00
7,8,Station Jean-Talon,06:06:00,06:06:00
8,9,Station Beaubien,06:08:00,06:08:00
9,10,Station Rosemont,06:09:00,06:09:00


In [24]:
def to_seconds(t):
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

one_trip["arr_sec"] = one_trip["arrival_time"].apply(to_seconds)
one_trip["travel_to_next"] = one_trip["arr_sec"].shift(-1) - one_trip["arr_sec"]

one_trip[["stop_sequence", "stop_name", "arrival_time", "arr_sec", "travel_to_next"]]

,stop_sequence,stop_name,arrival_time,arr_sec,travel_to_next
0,1,Station Montmorency -Zone B,05:54:00,21240,60.0
1,2,Station De la Concorde -Zone B,05:55:00,21300,180.0
2,3,Station Cartier -Zone B,05:58:00,21480,60.0
3,4,Station Henri-Bourassa,05:59:00,21540,120.0
4,5,Station Sauvé,06:01:00,21660,120.0
5,6,Station Crémazie,06:03:00,21780,120.0
6,7,Station Jarry,06:05:00,21900,60.0
7,8,Station Jean-Talon,06:06:00,21960,120.0
8,9,Station Beaubien,06:08:00,22080,60.0
9,10,Station Rosemont,06:09:00,22140,60.0


In [25]:
total = one_trip["arr_sec"].iloc[-1] - one_trip["arr_sec"].iloc[0]
print("Stations:", len(one_trip))
print("Total trip:", total / 60, "minutes")

one_trip.sort_values("travel_to_next", ascending=False).head(3)[["stop_name", "travel_to_next"]]

Stations: 31
Total trip: 46.0 minutes


,stop_name,travel_to_next
1,Station De la Concorde -Zone B,180.0
3,Station Henri-Bourassa,120.0
5,Station Crémazie,120.0


In [26]:
trip_stops = stops[stops["stop_id"].isin(one_trip["stop_id"])]
trip_stops.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding
25,36,10132,Station Lionel-Groulx,45.482509,-73.580180,https://www.stm.info/fr/infos/reseaux/metro/li...,0,STATION_M132,1
59,9999112,10146,Station Berri-UQAM,45.515242,-73.561025,https://www.stm.info/fr/infos/reseaux/metro/be...,0,STATION_M146,1
113,65,10222,Station Côte-Vertu,45.514254,-73.682752,https://www.stm.info/fr/infos/reseaux/metro/co...,0,STATION_M222,1
118,54,10224,Station Du Collège,45.509168,-73.674705,https://www.stm.info/fr/infos/reseaux/metro/du...,0,STATION_M224,1
122,53,10228,Station De la Savane,45.500087,-73.661580,https://www.stm.info/fr/infos/reseaux/metro/de...,0,STATION_M228,2


In [27]:
stops[stops["stop_name"].str.contains("Jean-Talon")]

,stop_id,stop_code,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding
194,9999052,10272,Station Jean-Talon,45.538911,-73.613941,https://www.stm.info/fr/infos/reseaux/metro/je...,0,STATION_M272,1
195,9999055,10272,Station Jean-Talon,45.539201,-73.615064,https://www.stm.info/fr/infos/reseaux/metro/je...,0,STATION_M272,1
196,05-01,10272,Station Jean-Talon - Accès Ouest rue Jean-Talon,45.538700,-73.613910,NaN,2,STATION_M272,1
197,05-02,10272,Station Jean-Talon - Accès Est rue Jean-Talon ...,45.540261,-73.613168,NaN,2,STATION_M272,2
198,05-03,10272,Station Jean-Talon - Accès av De Châteaubriand,45.539475,-73.612964,NaN,2,STATION_M272,2
...,...,...,...,...,...,...,...,...,...
5540,54990,54990,Jean-Talon / De Bellefeuille,45.590859,-73.576008,https://www.stm.info/fr/recherche#stq=54990,0,NaN,1
5541,54991,54991,Langelier / Jean-Talon,45.591997,-73.574550,https://www.stm.info/fr/recherche#stq=54991,0,NaN,1
5542,54993,54993,Langelier / Jean-Talon,45.592043,-73.575363,https://www.stm.info/fr/recherche#stq=54993,0,NaN,1
5543,54994,54994,Jean-Talon / Langelier,45.591862,-73.575093,https://www.stm.info/fr/recherche#stq=54994,0,NaN,1


In [28]:
import folium

# All platforms used by any metro trip, with their line
used = metro_stop_times[["trip_id", "stop_id"]].drop_duplicates("stop_id")
used = used.merge(metro_trips[["trip_id", "route_id"]], on="trip_id")
used = used.merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
used = used.merge(metro_routes[["route_id", "route_color"]], on="route_id")

m = folium.Map(location=[45.51, -73.60], zoom_start=11)

for _, row in used.iterrows():
    folium.CircleMarker(
        location=[float(row["stop_lat"]), float(row["stop_lon"])],
        radius=5,
        color="#" + row["route_color"],
        fill=True,
        tooltip=row["stop_name"],
    ).add_to(m)

m

In [29]:
rows = []
for route_id in ["1", "2", "4", "5"]:
    trip = metro_trips[(metro_trips["route_id"] == route_id) & (metro_trips["direction_id"] == "0")].iloc[0]
    st = metro_stop_times[metro_stop_times["trip_id"] == trip["trip_id"]].copy()
    st["stop_sequence"] = st["stop_sequence"].astype(int)
    st = st.sort_values("stop_sequence").merge(stops[["stop_id", "stop_name"]], on="stop_id")
    for _, r in st.iterrows():
        rows.append({"route_id": route_id, "seq": r["stop_sequence"], "stop_name": r["stop_name"], "x": "", "y": ""})

layout = pd.DataFrame(rows)
layout.to_csv("stations_schematic.csv", index=False)
layout.groupby("route_id").size()

route_id
1    27
2    31
4     3
5    12
dtype: int64

In [30]:
import math

WIDTH, HEIGHT, PAD = 1000, 1000, 80
ROTATE_DEG = 0    # try -35 to line up with Montreal's street grid ("Montreal north")
SPREAD = 0.55     # 1.0 = true geography; try 0.75 to give crowded downtown more room

# 1. One lat/lon per station name (platforms of the same station get averaged)
coords = stops[stops["stop_id"].isin(metro_stop_times["stop_id"])][["stop_name", "stop_lat", "stop_lon"]].copy()
coords[["stop_lat", "stop_lon"]] = coords[["stop_lat", "stop_lon"]].astype(float)
coords = coords.groupby("stop_name", as_index=False).mean()

layout = pd.read_csv("stations_schematic.csv", dtype=str)[["route_id", "seq", "stop_name"]]
layout = layout.merge(coords, on="stop_name", how="left")
print(layout["stop_lat"].isna().sum(), "stations missing coordinates")

# 2. Lat/lon -> flat x/y (longitude degrees shrink away from the equator)
lat0 = layout["stop_lat"].mean()
lon0 = layout["stop_lon"].mean()
x = (layout["stop_lon"] - lon0) * math.cos(math.radians(lat0))
y = -(layout["stop_lat"] - lat0)   # minus: screen y grows downward

# 3. Optional rotation
a = math.radians(ROTATE_DEG)
x, y = x * math.cos(a) - y * math.sin(a), x * math.sin(a) + y * math.cos(a)

# 4. Optional spread: push stations near the centre outward
r = (x**2 + y**2) ** 0.5
factor = (r / r.max()) ** (SPREAD - 1)
factor = factor.fillna(1)
x, y = x * factor, y * factor

# 5. Scale to fit the canvas, keeping proportions
scale = min((WIDTH - 2 * PAD) / (x.max() - x.min()), (HEIGHT - 2 * PAD) / (y.max() - y.min()))
layout["x"] = ((x - x.min()) * scale + PAD).round().astype(int)
layout["y"] = ((y - y.min()) * scale + PAD).round().astype(int)

layout[["route_id", "seq", "stop_name", "x", "y"]].to_csv("stations_schematic.csv", index=False)
layout[["stop_name", "x", "y"]].head(10)

0 stations missing coordinates


,stop_name,x,y
0,Station Angrignon,501,920
1,Station Monk,542,905
2,Station Jolicoeur,592,883
3,Station Verdun,636,871
4,Station De l'Église,660,858
5,Station LaSalle,673,821
6,Station Charlevoix,668,790
7,Station Lionel-Groulx,617,780
8,Station Atwater,590,748
9,Station Guy-Concordia,639,710


In [31]:
import json, datetime

def clean_name(n):
    return n.replace("Station ", "").replace(" -Zone B", "").strip()

today = datetime.date.today()
ymd = today.strftime("%Y%m%d")
weekday = today.strftime("%A").lower()

cal = pd.read_csv("gtfs/calendar.txt", dtype=str)
active = set(cal[(cal[weekday] == "1") & (cal["start_date"] <= ymd) & (cal["end_date"] >= ymd)]["service_id"])

# Exceptions: 1 = service added on this date, 2 = service removed
cd = pd.read_csv("gtfs/calendar_dates.txt", dtype=str)
cd_today = cd[cd["date"] == ymd]
active |= set(cd_today[cd_today["exception_type"] == "1"]["service_id"])
active -= set(cd_today[cd_today["exception_type"] == "2"]["service_id"])

today_trips = metro_trips[metro_trips["service_id"].isin(active)]

st = metro_stop_times[metro_stop_times["trip_id"].isin(today_trips["trip_id"])].copy()
st["stop_sequence"] = st["stop_sequence"].astype(int)
st["t"] = st["arrival_time"].apply(to_seconds)
st = st.merge(stops[["stop_id", "stop_name"]], on="stop_id")
st = st.merge(today_trips[["trip_id", "route_id"]], on="trip_id")
st = st.sort_values(["trip_id", "stop_sequence"])

out = []
for trip_id, g in st.groupby("trip_id"):
    out.append({
        "route": g["route_id"].iloc[0],
        "stops": [clean_name(n) for n in g["stop_name"]],
        "times": g["t"].tolist(),
    })

with open("trips_today.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False)

print(len(active), "active services,", len(out), "metro trips today")

9 active services, 1654 metro trips today


In [32]:
# Build a network model: stations, lines, and travel times between neighbours
from collections import defaultdict

# 1. Median travel time between each pair of adjacent stations, per line
seg_times = defaultdict(list)
for trip_id, g in st.groupby("trip_id"):
    route = g["route_id"].iloc[0]
    names = [clean_name(n) for n in g["stop_name"]]
    times = g["t"].tolist()
    for i in range(len(names) - 1):
        seg_times[(route, names[i], names[i + 1])].append(times[i + 1] - times[i])

edges = []
for (route, a, b), samples in seg_times.items():
    samples.sort()
    median = samples[len(samples) // 2]
    edges.append({"route": route, "from": a, "to": b, "seconds": int(median)})

# 2. Stations: position, lines served, terminals
layout = pd.read_csv("stations_schematic.csv", dtype=str)
routes_by_station = defaultdict(set)
for e in edges:
    routes_by_station[e["from"]].add(e["route"])
    routes_by_station[e["to"]].add(e["route"])

terminals = set()
for trip_id, g in st.groupby("trip_id"):
    terminals.add(clean_name(g["stop_name"].iloc[0]))
    terminals.add(clean_name(g["stop_name"].iloc[-1]))

stations = {}
for _, r in layout.drop_duplicates("stop_name").iterrows():
    name = clean_name(r["stop_name"])
    stations[name] = {
        "x": int(r["x"]),
        "y": int(r["y"]),
        "routes": sorted(routes_by_station[name]),
        "terminal": name in terminals,
    }

network = {
    "routes": {r["route_id"]: {"name": r["route_long_name"], "color": "#" + r["route_color"]}
               for _, r in metro_routes.iterrows()},
    "stations": stations,
    "edges": edges,
}

with open("network.json", "w", encoding="utf-8") as f:
    json.dump(network, f, ensure_ascii=False, indent=1)

interchanges = [n for n, s in stations.items() if len(s["routes"]) > 1]
print(len(stations), "stations,", len(edges), "edges")
print("Interchanges:", sorted(interchanges))
print("Terminals:", len(terminals))

68 stations, 138 edges
Interchanges: ['Berri-UQAM', 'Jean-Talon', 'Lionel-Groulx', 'Snowdon']
Terminals: 9


In [33]:
# Crop the map to the area the stations actually occupy
xs = [s["x"] for s in stations.values()]
ys = [s["y"] for s in stations.values()]
pad = 45
box = [min(xs) - pad, min(ys) - pad, max(xs) - min(xs) + 2 * pad, max(ys) - min(ys) + 2 * pad]

network["viewBox"] = " ".join(str(int(v)) for v in box)

with open("network.json", "w", encoding="utf-8") as f:
    json.dump(network, f, ensure_ascii=False, indent=1)

print("viewBox:", network["viewBox"])

viewBox: 35 35 894 930
